# Kauvar, Richman, Liu et al. 2025: mouse Neuropixels claims (DANDI:001326)

Paper: Kauvar, Richman, Liu et al., "Conserved brain-wide emergence of emotional response from sensation in humans and mice", Science 388, eadt3971 (2025), doi:10.1126/science.adt3971.

Data: the mouse Neuropixels recordings are not in dandiset 001347; the paper deposited them in DANDI:001326 (ref. 104), so these claims are `data: elsewhere`. DANDI:001326 has one published version, 0.250528.1957, which is pinned; the paper cites no version. Files are streamed with remfile and h5py (`dandi_io.open_h5`); nothing is downloaded. The Neuropixels-rig behaviour files of DANDI:001347 at 0.250528.0702 are read only to link sessions.

Text read by the claims file: the version of record with its supplement (Materials and Methods, figure legends, Table S3), as extracted in `paper/10.1126-science.adt3971.txt`. Methods details not in the claims file come from that text and from the mouse analysis code, Zenodo record 14876694 (`analysis.py`, md5 6cc7aa5059737c3bd753aad1f116019d), which is read, not imported; `mouse_npx.py` reimplements what is needed and its docstring says where every parameter comes from.

| claim | tested here |
|---|---|
| fig6d-mouse-persistent-activity-reduced | tested |
| fig6f-mouse-fast-activity-unchanged | tested |
| fig5e-mouse-emotion-dimension-late-reduced | tested |
| fig6h-mouse-timescale-shorter | tested |
| fig6j-mouse-coupling-reduced | tested |
| fig5d-mouse-emotion-dimension-accumulates | tested |
| figs34-mouse-tone-persistence-reduced | tested |
| fig4hi-mouse-rise-unchanged-decay-shortened | tested |
| fig4g-mouse-slow-clusters-altered | tested |
| fig5g-mouse-first-order-model | tested (neural fit on DANDI:001326, dose fits on DANDI:001347) |

In [ ]:
import os
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
import time, warnings
import numpy as np
import pandas as pd
import scipy.stats
import yaml
import matplotlib.pyplot as plt

import dandi_io
import mouse_npx as m

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)
pd.set_option('display.precision', 4)

with open('../../claims/001347/10.1126-science.adt3971.yaml') as fh:
    CLAIMS = {c['id']: c for c in yaml.safe_load(fh)['claims']}
def expected(cid):
    return [(e['quantity'], e['value'], e['tolerance']) for e in CLAIMS[cid].get('expected', [])]
def within(x, v, tol):
    return abs(x - v) <= tol

## Data check

Every NWB file of DANDI:001326 is streamed once. `mouse_npx.process_session` bins each unit's spikes at 10 ms, applies the 50-ms causal moving average, z-scores each unit over the recording, finds the puff-modulated cells, and keeps only the reduced arrays the claims need. This cell takes most of the run time.

In [ ]:
t0 = time.time()
paths = m.npx_paths()
Rs = []
with warnings.catch_warnings():
    warnings.simplefilter('ignore', RuntimeWarning)
    for p in paths:
        Rs.append(m.process_session(p))
print(f'{len(paths)} files read and reduced in {(time.time() - t0) / 60:.1f} min')
rows = []
for R in Rs:
    st = R['puff']
    rows.append(dict(session=f"sub-{R['subject']}_{R['session']}", mouse=R['subject'], probes=R['n_probes'],
                     units=R['n_units'], modulated=R['n_sel'], puff_series=st.shape[0],
                     puff_isi_s=np.median(np.diff(st, axis=1)),
                     series_before_injection=int(np.sum(st[:, 0] < R['ket_time'])),
                     tone_series=0 if R['tone'] is None else R['tone'].shape[0],
                     light_series=0 if R['light'] is None else R['light'].shape[0],
                     units_sorted_spikes=int(R['spikes_sorted'].sum())))
SESS = pd.DataFrame(rows)
print(SESS.to_string())
print('mice:', SESS.mouse.nunique(), 'sessions:', len(SESS))

In [ ]:
# Unit metadata against the Methods (Bombcell 'somatic' units, SNR > 2, > 500 spikes; ISI violations < 0.1)
q = np.concatenate([R['quality'] for R in Rs]); snr = np.concatenate([R['snr'] for R in Rs])
ns = np.concatenate([R['n_spikes'] for R in Rs])
print('units:', len(q), 'quality labels:', dict(zip(*np.unique(q, return_counts=True))))
print(f'SNR range {snr.min():.2f} to {snr.max():.1f}; n_spikes range {ns.min():.0f} to {ns.max():.0f}')
print('units with SNR <= 2:', int((snr <= 2).sum()), '; units with <= 500 spikes:', int((ns <= 500).sum()))
# stimulus durations and the injection time
dur = {}
for R in Rs:
    f = dandi_io.open_h5(R['path'], dandi_io.NPX_DANDISET, dandi_io.NPX_VERSION)
    st = f['intervals/Stimulus presentations/stimulus'][:].astype(str)
    d = f['intervals/Stimulus presentations/stop_time'][:] - f['intervals/Stimulus presentations/start_time'][:]
    for s in np.unique(st):
        dur.setdefault(s, []).append(d[st == s])
    f.close()
for s, v in dur.items():
    v = np.concatenate(v)
    print(f'{s}: {len(v)} presentations, duration {v.min():.3f} to {v.max():.3f} s')
gap = [R['ket_time'] - R['puff'][19, -1] for R in Rs]
nxt = [R['puff'][20, 0] - R['ket_time'] for R in Rs]
print(f"'Ketamine infusion start' falls {min(gap):.0f} to {max(gap):.0f} s after the last puff of series 20 "
      f"and {min(nxt):.0f} to {max(nxt):.0f} s before series 21")
print('modulated cells per session:', SESS.modulated.tolist(), '; fraction of units', round(SESS.modulated.sum() / SESS.units.sum(), 3))

Checks against the description:

- 13 sessions from 10 mice (subjects 0 to 9), as the paper states (Table S3, Fig. 4). Ten sessions have 4 probes; sub-5 ses-20230706 has 2 (the Methods say 3 to 4).
- Every unit is labelled 'good', with SNR above 2 and more than 500 spikes, the Methods' criteria; the ISI criterion cannot be checked because the files hold no ISI metric. The `n_spikes` column equals the number of stored spike times.
- Puffs, tones and lights last 250 ms, puffs come 3.0 s apart in series of 8 (the Methods give 250 ms and 3 s). Tones are in 9 sessions (the paper's tone analysis has 7 sessions from 7 mice).
- The 'Ketamine infusion start' timestamp falls between series 20 and 21 in every session (the Methods: infusion after the 20th trial).
- The spike times of every unit are stored unsorted (column `units_sorted_spikes` is 0 for all sessions), which breaks the NWB expectation that a unit's spike times are ascending; they are sorted here before binning.

### Matching sessions to the paper

The files carry subject numbers 0 to 9 and recording dates, not the paper's mouse or experiment identifiers. The Neuropixels-rig behaviour files in DANDI:001347 (named by experiment id, `sub-m<mouse>-<experiment>`) hold the trial onsets (`events-on`, 100 frames/s). The interval between series 20 and 21 contains the injection and differs from session to session; it identifies each behaviour file's Neuropixels session, and the whole interval sequence is then compared.

In [ ]:
beh = {}
for p in sorted(dandi_io.list_assets()):
    if p.startswith('sub-m') and not p.startswith('sub-mouse'):
        f = dandi_io.open_h5(p)
        beh[p.split('/')[0][4:]] = f['acquisition/events-on/data'][:] / f['acquisition/fps/data'][0]
        f.close()
rows = []
for name, on in beh.items():
    gb = np.diff(on)
    best, err = None, np.inf
    for R in Rs:
        gn = np.diff(R['puff'][:, 0])
        n = min(len(gb), len(gn))
        e = np.max(np.abs(gb[:n] - gn[:n]))
        if e < err:
            best, err, nn = R, e, n
    rows.append(dict(behaviour=name, trials=len(on), npx_session=f"sub-{best['subject']}_{best['session']}",
                     npx_series=best['puff'].shape[0], intervals_compared=nn, max_abs_diff_s=err,
                     gap_20_21_s=np.diff(on)[19]))
LINK = pd.DataFrame(rows)
print(LINK.to_string())
unmatched = sorted(set(SESS.session) - set(LINK.npx_session))
print('Neuropixels sessions without a behaviour file:', unmatched)
tone_sessions = SESS[SESS.tone_series > 0]
print('tone sessions:', len(tone_sessions), 'from', tone_sessions.mouse.nunique(), 'mice')
for mouse in ['1', '3', '5']:
    rest = tone_sessions[tone_sessions.mouse != mouse]
    rest_all = SESS[SESS.mouse != mouse]
    print(f'excluding mouse {mouse}: population {len(rest_all)} sessions / {rest_all.mouse.nunique()} mice; '
          f'tone {len(rest)} sessions / {rest.mouse.nunique()} mice')

The ten behaviour files match ten Neuropixels sessions, each with all its inter-series intervals equal within 0.1 s, so the clocks of the two files agree up to an offset. Experiment ids `m11208-959` and `m11208-962` are subject 1, `m11210-960` is subject 3's first session; sub-3 ses-20230225 and both sub-5 sessions have no behaviour file in 001347. The behaviour files name 9 mice (10 sessions), the Neuropixels files 10 mice (13 sessions).

The Methods exclude one animal "that exhibited notable stress response" from the cross-puff and population analyses, giving 11 sessions from 9 mice; neither the paper nor the code names it. Only mice 1, 3 and 5 have two sessions. Excluding mouse 5 is the only choice that also gives the tone analysis its 7 sessions from 7 mice (Table S3, fig. S34); excluding mouse 1 or 3 leaves 8 tone sessions. Mouse 5 is therefore taken as the excluded animal (an assumption; it also has the 2-probe session). The 13-session results are shown as a variant for the main claims. Fig. 4 analyses (clusters, rise and decay) use all 13 sessions, as Table S3 gives for Fig. 4.

In [ ]:
POP = [R for R in Rs if R['subject'] != '5']
TONE = [R for R in POP if R['tone'] is not None]
print('population analyses:', len(POP), 'sessions from', len({R['subject'] for R in POP}), 'mice')
print('tone analyses:', len(TONE), 'sessions from', len({R['subject'] for R in TONE}), 'mice')
# Clock check: mean preinfusion z-rate of all units around the puff (window -1 to +2 s)
t = (np.arange(m.WIN) - 100) * m.BIN
rows = []
for R in Rs:
    tr = R['pop_trace_all']
    base = tr[:100].mean(); sd = tr[:100].std()
    d = np.diff(tr)
    rows.append(dict(session=f"sub-{R['subject']}_{R['session']}",
                     resp_0_200ms=tr[100:120].mean() - base,
                     resp_shifted_minus_500ms=tr[50:70].mean() - base,
                     largest_rise_at_s=t[1:][np.argmax(d)], peak_at_s=t[np.argmax(tr)]))
print(pd.DataFrame(rows).to_string())

In every session the mean z-scored rate of all units rises steeply within the first tens of milliseconds after the stored puff time (largest one-bin rise and peak shortly after 0 s), and the same 200-ms window placed 500 ms earlier shows no response. The spike times and the stimulus table share a clock in every session.

## fig6d-mouse-persistent-activity-reduced

> In mice, activity along the persistent dimension (1 to 2 s after the puff) falls during ketamine (0.86 to 0.02) and partly recovers after (0.52).

Persistent dimension built from 0.5 to 1 s after each puff against the 1 s before it, over the 20 preinfusion series; the trial-averaged projection is z-scored over its 3-s window and the 1-s baseline subtracted (code); quantified 1 to 2 s. Readings run: the coding-dimension variance (Methods, per neuron 'M'; code, pooled 'C'), the epochs (Methods trials 22 to 32 and 34 to 40; code and Table S3 23 to 32 and 35 to 40), and the first puff of each series (code) or all 8 puffs. The main reading is M, first puff, code/Table S3 epochs: the Methods and Table S3 disagree on the trial counts and the code agrees with Table S3 (an assumption).

In [ ]:
cid = 'fig6d-mouse-persistent-activity-reduced'
def run_readings_3way(table_fn, readings, pop):
    rows = []
    for name, args in readings.items():
        d = table_fn(pop, *args)
        s = m.three_way_tests(d)
        row = dict(reading=name)
        for k in ('pre', 'ket', 'post'):
            mu, se = m.mean_sem(d[k]); row[k] = f'{mu:.3f} ± {se:.3f}'
        row.update({k: v for k, v in s.items()})
        rows.append(row)
    return pd.DataFrame(rows)

R6D = {}
for var in 'MC':
    for first in (True, False):
        for ename, (ket, post) in m.EPOCH_READINGS.items():
            R6D[f"var {var}, {'first puff' if first else 'all puffs'}, {ename}"] = (var, first, ket, post)
tab6d = run_readings_3way(m.persist_table, R6D, POP)
print(tab6d.to_string())
MAIN6D = ('M', True, 'ket_C', 'post_C')
d6d = m.persist_table(POP, *MAIN6D)
print('\nmain reading, per session:'); print(d6d.to_string())
s6d = m.three_way_tests(d6d)
print({k: round(v, 5) for k, v in s6d.items()})
for (qname, v, tol), col in zip(expected(cid), ('pre', 'ket', 'post')):
    mu, se = m.mean_sem(d6d[col])
    print(f'{qname}: {mu:.3f} ± {se:.3f} (expected {v} ± {tol}; within: {within(mu, v, tol)})')
dm = m.by_mouse(d6d, ['pre', 'ket', 'post'])
print('\naveraged within mouse (9 mice):', {k: round(v, 4) for k, v in m.three_way_tests(dm).items()})
d13 = m.persist_table(Rs, *MAIN6D)
print('all 13 sessions:', {k: (round(m.mean_sem(d13[k])[0], 3)) for k in ('pre', 'ket', 'post')},
      {k: round(v, 4) for k, v in m.three_way_tests(d13).items()})

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
for j, first in enumerate((True, False)):
    for k, col in (('pre', 'C0'), ('ket', 'C3'), ('post', 'C4')):
        tr = np.array([m.persist_projection(R, 'M', first, 'ket_C', 'post_C')[k] for R in POP])
        mu, ci = tr.mean(0), 1.96 * tr.std(0) / np.sqrt(len(tr))
        ax[j].plot(t, mu, color=col, label=k); ax[j].fill_between(t, mu - ci, mu + ci, color=col, alpha=.3)
    ax[j].set_title('first puff' if first else 'all 8 puffs'); ax[j].axvline(0, color='k', ls='--')
    ax[j].set_xlabel('time from puff (s)'); ax[j].set_ylabel('projection (z)')
ax[0].legend(); plt.tight_layout(); plt.show()

**Result: reproduced, resting on the assumption about the epochs.** Under the main reading (per-neuron variance, first puff, trials 23 to 32 and 35 to 40) the projection 1 to 2 s after the puff is 0.867 ± 0.059 before, -0.002 ± 0.133 during and 0.515 ± 0.219 after ketamine, each within its tolerance; pre against ketamine t = 5.48, corrected p = 0.0005; post against ketamine t = -2.54, corrected p = 0.029. The code's pooled variance gives the same picture (0.855, 0.019, 0.513; corrected p = 0.0004 and 0.026). With the Methods' trials (22 to 32, 34 to 40) the fall on ketamine holds (corrected p = 0.0008) but the recovery test is not significant with the first puff (corrected p = 0.17; 0.017 when all 8 puffs are averaged). Averaged within mouse (9 mice) the two tests give corrected p = 0.0033 and 0.030; with all 13 sessions 0.0001 and 0.032.

## fig6f-mouse-fast-activity-unchanged

> In mice, activity along the fast dimension (0 to 70 ms) is not changed by ketamine.

Fast dimension built from 0 to 70 ms after each puff (all 8 puffs of the 20 preinfusion series) against the 1 s before it (Methods, 'reference window (average activity in the 1s before each puff)'; the code uses the 70 ms before, a variant); projection averaged over all puffs, z-scored, quantified 0 to 70 ms; epochs as in fig6d.

In [ ]:
cid = 'fig6f-mouse-fast-activity-unchanged'
R6F = {}
for var in 'MC':
    for ref in ('1s', '70'):
        for ename, (ket, post) in m.EPOCH_READINGS.items():
            R6F[f"var {var}, reference {ref}, {ename}"] = (var, ref, ket, post)
tab6f = run_readings_3way(m.fast_table, R6F, POP)
print(tab6f.to_string())
MAIN6F = ('M', '1s', 'ket_C', 'post_C')
d6f = m.fast_table(POP, *MAIN6F)
print('\nmain reading, per session:'); print(d6f.to_string())
s6f = m.three_way_tests(d6f)
print({k: round(v, 5) for k, v in s6f.items()})
for (qname, v, tol), col in zip(expected(cid), ('pre', 'ket', 'post')):
    mu, se = m.mean_sem(d6f[col])
    print(f'{qname}: {mu:.3f} ± {se:.3f} (expected {v} ± {tol}; within: {within(mu, v, tol)})')
for a, b in (('pre', 'ket'), ('post', 'ket')):
    diff = d6f[a] - d6f[b]
    print(f'{a} - {b}: mean {diff.mean():.3f}, 95% CI {diff.mean() - 1.96 * diff.std(ddof=1) / np.sqrt(len(diff)):.3f} '
          f'to {diff.mean() + 1.96 * diff.std(ddof=1) / np.sqrt(len(diff)):.3f}, Cohen dz {diff.mean() / diff.std(ddof=1):.2f}')
dm = m.by_mouse(d6f, ['pre', 'ket', 'post'])
print('averaged within mouse (9 mice):', {k: round(v, 4) for k, v in m.three_way_tests(dm).items()})
d13 = m.fast_table(Rs, *MAIN6F)
print('all 13 sessions:', {k: round(v, 4) for k, v in m.three_way_tests(d13).items()})

**Result: not reproduced.** This is a claim of absence, judged on the paper's two tests, and one of them finds a difference. Pre against ketamine is not significant in any reading (main reading t = -0.43, corrected p = 0.67; difference -0.054, 95% CI -0.296 to 0.189, dz = -0.13), as the paper reports. Ketamine against post is significant in every reading (main reading t = 4.07, corrected p = 0.0045; post below ketamine by 0.322, dz = -1.23), where the paper reports corrected p = 0.066; the criterion asks both tests to be non-significant. The main reading's means (4.226, 4.279, 3.957) are below the paper's by more than its s.e.m.; the code's reading (pooled variance, 70-ms reference) gives 4.433, 4.496 and 4.161, within them.

## fig5e-mouse-emotion-dimension-late-reduced

> Along the mouse emotion-like dimension, ketamine leaves the rapid response to the first puff unchanged (0 to 0.25 s) but removes the late response (2 to 3 s).

Emotion-like dimension per session from the puff-modulated cells: target window 1 s starting 5 s after the last puff onset of each preinfusion series against the 750 ms ending 250 ms before the first puff, orthogonalised (QR) against a fast dimension (200 ms after against 200 ms before puffs 5 to 8). Signal: the projection of the trial-averaged series, baseline the 250 ms before the first puff; early 0 to 0.25 s, late 2 to 3 s after the first puff; paired t-tests with Benjamini-Hochberg over the two. Readings: variance M or C; fast dimension from puffs 5 to 8 or from the last 4 preinfusion series; ketamine trials 22 to 32 or 23 to 32. A further variant uses the code's target window, bins 400:500 of a window that, from the code's plotting offsets, starts 1 s before the last puff (3 to 4 s after the last puff onset).

In [ ]:
cid = 'fig5e-mouse-emotion-dimension-late-reduced'
rows = []
TAB5E = {}
for target in ('5-6', '3-4'):
    for var in 'MC':
        for eye in ('puffs5-8', 'last4series'):
            for ket in ('ket_C', 'ket_M'):
                d, sig, tt = m.affective_table(POP, target, var, eye, ket)
                TAB5E[(target, var, eye, ket)] = (d, sig, tt)
                te, pe = m.paired(d['pre_early'], d['ket_early']); tl, pl = m.paired(d['pre_late'], d['ket_late'])
                q = m.bh([pe, pl])
                r = dict(target=target + ' s', var=var, fast_dim=eye, ket=ket)
                for c in ('pre_early', 'ket_early', 'pre_late', 'ket_late'):
                    mu, se = m.mean_sem(d[c]); r[c] = f'{mu:.3f} ± {se:.3f}'
                r.update(t_early=te, q_early=q[0], t_late=tl, q_late=q[1])
                rows.append(r)
print(pd.DataFrame(rows).to_string())
MAIN5E = ('5-6', 'M', 'puffs5-8', 'ket_C')
d5e = TAB5E[MAIN5E][0]
print('\nmain reading, per session:'); print(d5e.to_string())
te, pe = m.paired(d5e['pre_early'], d5e['ket_early']); tl, pl = m.paired(d5e['pre_late'], d5e['ket_late'])
print(f'early t = {te:.3f}, late t = {tl:.3f}, BH-corrected p = {np.round(m.bh([pe, pl]), 4)}')
for (qname, v, tol), col in zip(expected(cid), ('pre_early', 'ket_early', 'pre_late', 'ket_late')):
    mu, se = m.mean_sem(d5e[col])
    print(f'{qname}: {mu:.3f} ± {se:.3f} (expected {v} ± {tol}; within: {within(mu, v, tol)})')
CODE5E = ('3-4', 'C', 'puffs5-8', 'ket_C')
dc = TAB5E[CODE5E][0]
te, pe = m.paired(dc['pre_early'], dc['ket_early']); tl, pl = m.paired(dc['pre_late'], dc['ket_late'])
print(f'\ncode-window variant (3-4 s, pooled variance): early t = {te:.3f}, late t = {tl:.3f}, BH p = {np.round(m.bh([pe, pl]), 4)}')
for (qname, v, tol), col in zip(expected(cid), ('pre_early', 'ket_early', 'pre_late', 'ket_late')):
    mu, se = m.mean_sem(dc[col]); print(f'  {qname}: {mu:.3f} ± {se:.3f} (expected {v} ± {tol}; within: {within(mu, v, tol)})')
dm = m.by_mouse(d5e, ['pre_early', 'ket_early', 'pre_late', 'ket_late'])
print('main reading averaged within mouse: late t, p =', np.round(m.paired(dm['pre_late'], dm['ket_late']), 4),
      '; early t, p =', np.round(m.paired(dm['pre_early'], dm['ket_early']), 4))
d13, _, _ = m.affective_table(Rs, *MAIN5E)
print('main reading, 13 sessions: late t, p =', np.round(m.paired(d13['pre_late'], d13['ket_late']), 4),
      '; early t, p =', np.round(m.paired(d13['pre_early'], d13['ket_early']), 4))

**Result: not reproduced** with the dimension the criterion specifies (target 1 s starting 5 s after the last puff). Late activity falls from 0.343 ± 0.143 to 0.009 ± 0.076, but the late test gives t = 1.90, corrected p = 0.086, and the early test t = 1.95, corrected p = 0.086; the preinfusion means (0.899 early, 0.343 late) are outside their tolerances. No reading with the 5 to 6 s window gives a corrected late p below 0.05 (0.066 to 0.158). With the window the code uses (bins 400:500 of a window that the plotting offsets place 1 s before the last puff, so 3 to 4 s after it) and the pooled variance, the paper's values return: early 0.608 and 0.308 (corrected p = 0.22), late 0.534 and 0.004 (t = 2.77, corrected p = 0.039), all four means within tolerance.

## fig5d-mouse-emotion-dimension-accumulates

> In mice, activity along the emotion-like dimension rises and saturates across the eight-puff series and decays over tens of seconds after it; on ketamine it does not accumulate.

Signal of fig5e (each session on its own dimension, baseline the 250 ms before the first puff), mean and 95% interval (1.96 s.e.m.) over the 11 sessions; the 23 to 25 s stretch around the training window is left out, as in the code. Criterion values: the 1 s before the eighth puff (20 to 21 s) minus the 1 s before the second puff (2 to 3 s), and the mean 10 to 20 s after the last puff (31 to 41 s after the first).

In [ ]:
cid = 'fig5d-mouse-emotion-dimension-accumulates'
rows = []
for key, (d, sig, tt) in TAB5E.items():
    r = dict(target=key[0], var=key[1], fast_dim=key[2], ket=key[3])
    for k in ('pre', 'ket'):
        S = sig[k]
        w = lambda a, b: np.nanmean(S[:, (tt >= a) & (tt < b)], 1)
        acc = w(20, 21) - w(2, 3); late = w(31, 41)
        r[f'{k}_accum'] = acc.mean(); r[f'{k}_accum_ci'] = 1.96 * acc.std() / np.sqrt(len(acc))
        r[f'{k}_10_20s'] = late.mean(); r[f'{k}_10_20s_ci'] = 1.96 * late.std() / np.sqrt(len(late))
    r['pass'] = (r['pre_accum'] > 0) and (r['pre_10_20s'] - r['pre_10_20s_ci'] > 0) and (r['ket_accum'] <= r['ket_accum_ci'])
    rows.append(r)
tab5d = pd.DataFrame(rows)
print(tab5d.to_string())
d, sig, tt = TAB5E[MAIN5E]
for k in ('pre', 'ket'):
    S = sig[k]
    acc = np.nanmean(S[:, (tt >= 20) & (tt < 21)], 1) - np.nanmean(S[:, (tt >= 2) & (tt < 3)], 1)
    print(k, 'per session accumulation:', np.round(acc, 3))
fig, ax = plt.subplots(figsize=(9, 3))
gapless = ~np.isnan(sig['pre'][0])
for k, col in (('pre', 'C0'), ('ket', 'C3')):
    S = sig[k][:, gapless]; mu = S.mean(0); ci = 1.96 * S.std(0) / np.sqrt(len(S))
    ax.plot(tt[gapless][tt[gapless] < 23], mu[tt[gapless] < 23], color=col, label=k)
    ax.plot(tt[gapless][tt[gapless] >= 25], mu[tt[gapless] >= 25], color=col)
    ax.fill_between(tt[gapless], mu - ci, mu + ci, color=col, alpha=.3)
for i in range(8):
    ax.axvline(3 * i, color='k', ls='--', alpha=.3)
ax.axhline(0, color='k'); ax.set_xlabel('time from first puff (s)'); ax.set_ylabel('emotion-like dim. (a.u.)'); ax.legend()
plt.show()
print('plotted session means range from', min(sig[k][:, gapless].mean(0).min() for k in sig).round(3), 'to',
      max(sig[k][:, gapless].mean(0).max() for k in sig).round(3), '; all within the axes')

**Result: reproduced.** Before ketamine the activity in the 1 s before the eighth puff exceeds that before the second by 0.846 (95% interval ± 0.170), every session positive (0.354 to 1.291), and 10 to 20 s after the last puff it is 0.838 ± 0.169, above zero; on ketamine the accumulation is -0.048 ± 0.114. All 16 readings pass the three conditions.

## fig6h-mouse-timescale-shorter

> In mice, the baseline intrinsic timescale of the persistent population is shorter during ketamine (0.14 s) than before (0.45 s).

The 30 s before each series projected on the persistent dimension (all puff-modulated cells, session z-scored rates); autocorrelation fitted with a*exp(-x/b)+c over 2.5 s from a = 1, b = 0.2, c = 0. Series: the code's 11 to 20 and 23 to 32 (Table S3: 10 and 10; the Methods do not say which) and, as a variant, 1 to 20 and 22 to 32. Readings: fit from lag 0 (Methods, "from the peak") or lag one bin (code); each series fitted and the timescales averaged (Methods, Table S3) or the series concatenated (code); variance M or C.

In [ ]:
cid = 'fig6h-mouse-timescale-shorter'
SER6H = {'code 11-20 / 23-32': (np.arange(10, 20), np.arange(22, 32)),
         'Methods epochs 1-20 / 22-32': (np.arange(0, 20), np.arange(21, 32))}
rows = []
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for var in 'MC':
        for sname, (sp, sk) in SER6H.items():
            for lag in (0, 1):
                for mode in ('per-series', 'concat'):
                    d, nf = m.timescale_table(POP, var, sp, sk, lag, mode)
                    tt_, pp_ = m.paired(d['pre'], d['ket'])
                    r = dict(var=var, series=sname, lag=lag, mode=mode, failed_fits=nf)
                    for k in ('pre', 'ket'):
                        mu, se = m.mean_sem(d[k]); r[k] = f'{mu:.3f} ± {se:.3f}'
                    r.update(t=tt_, p=pp_); rows.append(r)
    print(pd.DataFrame(rows).to_string())
    d6h, nf = m.timescale_table(POP, 'M', *SER6H['code 11-20 / 23-32'], 0, 'per-series')
print('\nmain reading (M, code series, lag 0, per series), per session:'); print(d6h.to_string())
print('failed fits:', nf, '; paired t, p =', np.round(m.paired(d6h['pre'], d6h['ket']), 4))
for (qname, v, tol), col in zip(expected(cid), ('pre', 'ket')):
    mu, se = m.mean_sem(d6h[col]); print(f'{qname}: {mu:.3f} ± {se:.3f} s (expected {v} ± {tol}; within: {within(mu, v, tol)})')
dm = m.by_mouse(d6h, ['pre', 'ket'])
print('averaged within mouse:', np.round(m.paired(dm['pre'], dm['ket']), 4), '; sessions with ketamine shorter:', int((d6h.ket < d6h.pre).sum()), 'of', len(d6h))

**Result: reproduced.** Main reading: 0.437 ± 0.077 s before and 0.162 ± 0.079 s during ketamine, both within tolerance; t = 3.51, p = 0.0057; 9 of 11 sessions shorter on ketamine; no fit failed. All 16 readings give p < 0.05 (0.004 to 0.038) with ketamine shorter; their preinfusion means range from 0.323 to 0.557 s and some fall outside the tolerance. Averaged within mouse, t = 3.14, p = 0.014.

## fig6j-mouse-coupling-reduced

> In mice, baseline pairwise correlation among the most persistent neurons halves during ketamine (0.49 of preinfusion).

Pearson correlations of the session z-scored rates in the 30 s before each series, among the puff-modulated cells whose persistent weight is above the 90th percentile of |w|. Readings: selection and average (Methods: |w| above the percentile and the mean of r; code: signed w above it and the median of |r|); correlations within each 30-s window averaged over series (Methods, Table S3: "Values averaged within session across trials") or over the concatenated windows (code); variance M or C; series of the code (1 to 10, 23 to 32; Table S3: 10 and 10) or of the Methods epochs (1 to 20, 22 to 32). The test is the code's: paired t-test of the normalised values (1 against ketamine/preinfusion).

In [ ]:
cid = 'fig6j-mouse-coupling-reduced'
SER6J = {'code 1-10 / 23-32': ('pre_C', 'ket_C', np.arange(0, 10), np.arange(22, 32)),
         'Methods epochs 1-20 / 22-32': ('pre_M', 'ket_M', np.arange(0, 20), np.arange(21, 32))}
def coupling_table(pop, var, rule, sname, mode):
    ep_pre, ep_ket, sp, sk = SER6J[sname]
    rows = []
    for R in pop:
        if mode == 'per-series':
            v = R['coupling_series'][(var, rule)]
            rows.append(dict(pre=np.nanmean(v[sp]), ket=np.nanmean(v[sk]), n_cells=R['coupling'][(var, rule, ep_pre, 'n_cells')]))
        else:
            rows.append(dict(pre=R['coupling'][(var, rule, ep_pre)], ket=R['coupling'][(var, rule, ep_ket)],
                             n_cells=R['coupling'][(var, rule, ep_pre, 'n_cells')]))
    d = m._df(pop, rows); d['ratio'] = d.ket / d.pre
    return d
rows = []
for var in 'MC':
    for rule in 'MC':
        for sname in SER6J:
            for mode in ('per-series', 'concat'):
                d = coupling_table(POP, var, rule, sname, mode)
                mu, se = m.mean_sem(d.ratio); tt_, pp_ = m.paired(np.ones(len(d)), d.ratio)
                rows.append(dict(var=var, rule=rule, series=sname, mode=mode, ratio=f'{mu:.3f} ± {se:.3f}', t=tt_, p=pp_,
                                 p_raw_pre_vs_ket=m.paired(d.pre, d.ket)[1]))
print(pd.DataFrame(rows).to_string())
d6j = coupling_table(POP, 'M', 'M', 'code 1-10 / 23-32', 'per-series')
print('\nmain reading (M weights, |w| and mean r, code series, per series), per session:'); print(d6j.to_string())
mu, se = m.mean_sem(d6j.ratio)
print(f'ratio {mu:.3f} ± {se:.3f}; paired t, p =', np.round(m.paired(np.ones(len(d6j)), d6j.ratio), 5))
for (qname, v, tol) in expected(cid):
    print(f'{qname}: {mu:.3f} (expected {v} ± {tol}; within: {within(mu, v, tol)})')
dc = coupling_table(POP, 'C', 'C', 'code 1-10 / 23-32', 'concat')
print('code reading (C weights, signed w, median |r|, concatenated):', np.round(m.mean_sem(dc.ratio), 3), np.round(m.paired(np.ones(len(dc)), dc.ratio), 6))
dm = m.by_mouse(d6j, ['ratio'])
print('main reading averaged within mouse: mean ratio', round(dm.ratio.mean(), 3), np.round(m.paired(np.ones(len(dm)), dm.ratio), 4))

**Result: not reproduced** under the Methods' reading. With the per-neuron-variance dimension, cells with |w| above the 90th percentile and the mean Pearson r, the ketamine/preinfusion ratio is 0.852 ± 0.128 (t = 1.10, p = 0.30), outside 0.49 ± 0.07; 9 of 11 sessions are below 1. The outcome turns on the dimension's variance: with the code's pooled variance every reading over the code's series gives a reduction (the code's full reading 0.559 ± 0.067, t = 6.32, p = 8.7e-05, within tolerance; with the Methods' selection and averaging 0.576 ± 0.101, p = 0.0025). With per-neuron variance only the code's selection (signed w, median |r|) gives a reduction (0.832, p = 0.005; 0.724, p = 0.012), outside tolerance. Over the Methods' series (1 to 20, 22 to 32) the concatenated readings show no reduction.

## figs34-mouse-tone-persistence-reduced

> For a neutral tone, mice do not close their eyes; fast tone activity is unchanged by ketamine, while persistent tone activity, its baseline timescale and its baseline coupling are all reduced.

Seven tone sessions from seven mice (mouse 5 excluded). Tone dimensions are built from the tone series before ketamine on the puff-modulated cells (code): fast 0 to 70 ms against the 1 s before (Methods; code 70 ms, a variant), persistent 150 to 350 ms against the 1 s before, quantified 350 to 500 ms, all 8 tones averaged. Epochs: up to 10 tone series before the split, tone series 2 to 6 after it (ketamine), 8 to 10 after it (post) (code; Table S3 10, 5, 3). Split at the code's ketamine start (30 s after the last puff of series 20: 9 preinfusion tone series) or at the file's injection time (10, as Table S3). Timescale and coupling use the 30 s before each tone series (Methods), with the code's windows before puff series as a coupling variant. The eye-closure part of the statement is not part of the criterion and is not tested: the behaviour files hold puff trials only.

In [ ]:
cid = 'figs34-mouse-tone-persistence-reduced'
def tone_tables(pop, split, var, ref):
    fr, pr = [], []
    for R in pop:
        n = R['n_sel']; W = R['tone_win'][split]
        w = m.cd_weights(W['f70'].reshape(n, -1), (W['b1s'] if ref == '1s' else W['b70']).reshape(n, -1), var)
        wp = R['cd_tone_persist'][split][var]
        fr.append({k: m.zproj(np.tensordot(w, R['tone_ep'][split][k], axes=([0], [0])).mean(0))[100:107].mean() for k in ('pre', 'ket', 'post')})
        pr.append({k: m.zproj(np.tensordot(wp, R['tone_ep'][split][k], axes=([0], [0])).mean(0))[135:150].mean() for k in ('pre', 'ket', 'post')})
    return m._df(pop, fr), m._df(pop, pr)
rows = []
for split in ('code', 'inj'):
    for var in 'MC':
        for ref in ('1s', '70'):
            df_, dp_ = tone_tables(TONE, split, var, ref)
            sf, sp_ = m.three_way_tests(df_), m.three_way_tests(dp_)
            r = dict(split=split, var=var, fast_ref=ref)
            for k in ('pre', 'ket', 'post'):
                r['fast_' + k] = '%.3f ± %.3f' % m.mean_sem(df_[k])
            r.update(fast_q_pre_ket=sf['q_pre_ket'], fast_q_ket_post=sf['q_ket_post'])
            for k in ('pre', 'ket', 'post'):
                r['pers_' + k] = '%.3f ± %.3f' % m.mean_sem(dp_[k])
            r.update(pers_t_pre_ket=sp_['t_pre_ket'], pers_q_pre_ket=sp_['q_pre_ket'], pers_t_ket_post=sp_['t_ket_post'], pers_q_ket_post=sp_['q_ket_post'])
            rows.append(r)
print(pd.DataFrame(rows).to_string())
MAINS34 = ('inj', 'M', '1s')
dfs34, dps34 = tone_tables(TONE, *MAINS34)
print('\nmain reading (injection split, M, 1-s reference): fast', {k: round(v, 4) for k, v in m.three_way_tests(dfs34).items()})
print('persistent', {k: round(v, 4) for k, v in m.three_way_tests(dps34).items()})
print(dps34.to_string())

In [ ]:
rows = []
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for split in ('code', 'inj'):
        for var in 'MC':
            for lag in (0, 1):
                for mode in ('per-series', 'concat'):
                    d, nf = m.timescale_table(TONE, var, lambda R: R['tone_sets'][split]['pre'],
                                              lambda R: R['tone_sets'][split]['ket'], lag, mode,
                                              key=lambda R: R['tone_base_proj'][split])
                    r = dict(split=split, var=var, lag=lag, mode=mode, failed=nf, pre='%.3f ± %.3f' % m.mean_sem(d.pre),
                             ket='%.3f ± %.3f' % m.mean_sem(d.ket))
                    r['t'], r['p'] = m.paired(d.pre, d.ket); rows.append(r)
    print(pd.DataFrame(rows).to_string())
    dts34, _ = m.timescale_table(TONE, 'M', lambda R: R['tone_sets']['inj']['pre'], lambda R: R['tone_sets']['inj']['ket'], 0,
                                 'per-series', key=lambda R: R['tone_base_proj']['inj'])
print('\nmain reading timescales:'); print(dts34.to_string()); print('t, p =', np.round(m.paired(dts34.pre, dts34.ket), 4))
rows = []
for split in ('code', 'inj'):
    for var in 'MC':
        for rule in 'MC':
            for mode in ('per-series', 'concat', 'code puff windows'):
                vals = []
                for R in TONE:
                    ts = R['tone_sets'][split]
                    if mode == 'per-series':
                        v = R['tone_coupling_series'][split][(var, rule)]; vals.append((np.nanmean(v[ts['pre']]), np.nanmean(v[ts['ket']])))
                    elif mode == 'concat':
                        c = R['tone_coupling'][split]; vals.append((c[(var, rule, 'pre')], c[(var, rule, 'ket')]))
                    else:
                        c = R['tone_coupling_puffwin'][split]; vals.append((c[(var, rule, 'pre_C')], c[(var, rule, 'ket_C')]))
                vals = np.array(vals); ratio = vals[:, 1] / vals[:, 0]
                tt_, pp_ = m.paired(np.ones(len(ratio)), ratio)
                rows.append(dict(split=split, var=var, rule=rule, windows=mode, ratio='%.3f ± %.3f' % m.mean_sem(ratio), t=tt_, p=pp_))
    COUP34 = pd.DataFrame(rows)
print(COUP34.to_string())
ex = expected(cid)
print('\nmain reading against expected values:')
vals = [m.mean_sem(dps34.pre)[0], m.mean_sem(dps34.ket)[0], m.mean_sem(dts34.pre)[0], m.mean_sem(dts34.ket)[0]]
main_c = COUP34[(COUP34['split'] == 'inj') & (COUP34['var'] == 'M') & (COUP34['rule'] == 'M') & (COUP34['windows'] == 'per-series')].iloc[0]
vals.append(float(main_c.ratio.split(' ± ')[0]))
for (qname, v, tol), x in zip(ex, vals):
    print(f'{qname}: {x:.3f} (expected {v} ± {tol}; within: {within(x, v, tol)})')
print('main coupling test:', main_c.to_dict())

**Result: partially reproduced.** Fast tone activity is not significantly changed (main reading corrected p = 0.16 pre against ketamine and 0.43 post against ketamine; every reading above 0.08). Persistent tone activity falls from 1.639 ± 0.053 to 0.481 ± 0.162 (t = 7.30, corrected p = 0.0007) and recovers (t = -2.91, corrected p = 0.027); the preinfusion mean is above 1.531 ± 0.081, while the code's pooled variance gives 1.512 to 1.522, within it. The tone timescale falls from 0.304 ± 0.043 to 0.115 ± 0.027 s (t = 5.43, p = 0.0016); the preinfusion value is within 0.268 ± 0.037 and the ketamine value outside 0.089 ± 0.012. Coupling among the top tone-dimension cells is not reduced under the Methods' reading (ratio 1.238 ± 0.451, p = 0.64); a reduction appears only with the code's pooled variance and median |r|, strongest over the windows before puff series that the code uses (0.535 ± 0.104, p = 0.006).

## fig4g-mouse-slow-clusters-altered

> Ketamine leaves neuron clusters with rapid preinfusion puff responses little changed and substantially alters clusters with slower, more persistent responses.

All 13 sessions. Each puff-modulated cell's preinfusion per-puff trace (mean over the 20 series and 8 puffs, baseline subtracted) enters Scanpy (PCA arpack, 30 neighbours on 20 PCs, UMAP, Leiden with leidenalg at resolution 1). Per cluster, cells' traces before and during ketamine (trials as in fig6d), z-scored within cluster and condition, mean and 95% interval over cells; the measure is the fraction of 0 to 2 s after the puff in which the two intervals do not overlap; clusters ordered by the time their preinfusion average decays to 25% of its peak (code find_decay).

In [ ]:
cid = 'fig4g-mouse-slow-clusters-altered'
t0 = time.time()
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    LAB, SESSIX = m.cluster_cells(Rs)
print(f'clustering {len(LAB)} cells: {len(np.unique(LAB))} clusters in {time.time() - t0:.0f} s; sizes {np.bincount(LAB).tolist()}')
mice_of = np.array([Rs[i]['subject'] for i in SESSIX])
print('mice contributing to each cluster:', [len(np.unique(mice_of[LAB == c])) for c in np.unique(LAB)])
def cluster_measure(lab, ket):
    tr = m.cluster_condition_traces(Rs, lab, ket)
    rows = []
    for c, d in tr.items():
        mp, cp, n = d['pre']; mk, ck, _ = d['ket']
        rows.append(dict(cluster=c, cells=n, decay_s=m.find_decay(mp), nonoverlap_0_2s=np.mean((np.abs(mp - mk) > (cp + ck))[100:300])))
    d = pd.DataFrame(rows).sort_values('decay_s').reset_index(drop=True)
    rho, p = scipy.stats.spearmanr(d.decay_s, d.nonoverlap_0_2s)
    return d, rho, p, d.nonoverlap_0_2s.iloc[-3:].mean(), d.nonoverlap_0_2s.iloc[:3].mean(), tr
for ket in ('ket_C', 'ket_M'):
    d, rho, p, slow, fast, tr = cluster_measure(LAB, ket)
    print(f'\nketamine trials {ket}:'); print(d.to_string())
    print(f'Spearman rho {rho:.3f} (p {p:.3f}); slowest 3 {slow:.3f}, fastest 3 {fast:.3f}')
    if ket == 'ket_C':
        D4G, TR4G = d, tr
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for res in (1.2, 1.5):
        lab2, _ = m.cluster_cells(Rs, resolution=res)
        d, rho, p, slow, fast, _ = cluster_measure(lab2, 'ket_C')
        print(f'resolution {res}: {len(np.unique(lab2))} clusters, Spearman rho {rho:.3f} (p {p:.3f}); slowest 3 {slow:.3f}, fastest 3 {fast:.3f}')

In [ ]:
fig, axs = plt.subplots(2, 6, figsize=(14, 4.5), sharex=True)
for ax, (_, row) in zip(axs.ravel(), D4G.iterrows()):
    d = TR4G[row.cluster]
    for k, col in (('pre', 'C0'), ('ket', 'C3')):
        mu, ci, _ = d[k]; ax.plot(t, mu, color=col); ax.fill_between(t, mu - ci, mu + ci, color=col, alpha=.3)
    ax.set_title(f"cl {row.cluster}: decay {row.decay_s:.2f} s", fontsize=8); ax.set_xlim(-0.5, 2)
plt.tight_layout(); plt.show()

**Result: reproduced.** The 6553 modulated cells of 13 sessions form 11 clusters (the paper reports 12), each with cells from all 10 mice. With ketamine trials 23 to 32, the fraction of 0 to 2 s in which the two intervals separate rises with the preinfusion decay time (Spearman rho = 0.47, p = 0.14), and the three slowest clusters average 0.507 against 0.368 for the three fastest. With trials 22 to 32 rho = 0.27 (0.475 against 0.355); at resolutions giving 13 and 17 clusters rho = 0.54 and 0.33, slowest above fastest in both. The correlation is not significant in any version, and the three fastest clusters still separate over 0.255 to 0.515 of the window.

## fig4hi-mouse-rise-unchanged-decay-shortened

> Ketamine leaves the rise times of puff-evoked regional activity unchanged but shortens regional decay times; decay times run from subcortical to frontal regions.

All 13 sessions, clusters of fig4g. Per region (layer suffixes merged, code reduce_region; at least 30 modulated cells; 'NA', 'STR', 'MB', 'HY' left out as in the code's plots), cluster averages of the first two puffs of each series (40 preinfusion replicates, 20 ketamine), d' against the 1-s baseline, rise to and decay to 25% of the normalised peak, peaks after 750 ms dropped, cluster-weighted; 1000 bootstrap resamples of the puff replicates (seed 0). Region filter on peak d' 1.3 (Methods, code) or 2 (Table S3). Two readings of "its bootstrap 95% interval": the interval of the median over regions across bootstrap samples, or the 2.5 to 97.5 percentiles of the code's pooled distribution of differences over regions and bootstrap samples.

In [ ]:
cid = 'fig4hi-mouse-rise-unchanged-decay-shortened'
rows = []
t0 = time.time()
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for ket_name, ket_series in (('code 23-32', np.arange(22, 32)), ('Methods 22-32', np.arange(21, 32))):
        for thr in (1.3, 2.0):
            res = m.region_rise_decay(Rs, LAB, ket_series, dprime_min=thr, n_boot=1000, seed=0)
            regs = list(res)
            D = np.stack([res[r]['ket'][1] - res[r]['pre'][1] for r in regs])
            RR = np.stack([res[r]['ket'][0] - res[r]['pre'][0] for r in regs])
            md_, mr_ = np.nanmedian(D, 0), np.nanmedian(RR, 0)
            rows.append(dict(ket=ket_name, dprime_min=thr, regions=len(regs), decay_median=np.nanmedian(D),
                             decay_ci_of_median=np.round(np.percentile(md_, [2.5, 97.5]), 4).tolist(),
                             decay_pooled_95=np.round(np.nanpercentile(D, [2.5, 97.5]), 4).tolist(),
                             rise_median=np.nanmedian(RR), rise_ci_of_median=np.round(np.percentile(mr_, [2.5, 97.5]), 4).tolist(),
                             rise_pooled_95=np.round(np.nanpercentile(RR, [2.5, 97.5]), 4).tolist()))
            if ket_name == 'code 23-32' and thr == 1.3:
                RES4H = res
print(f'{time.time() - t0:.0f} s')
TAB4H = pd.DataFrame(rows); print(TAB4H.to_string())
reg_rows = [dict(region=r, cells=v['n'], peak_dprime=v['dmax'], pre_rise=np.nanmedian(v['pre'][0]), ket_rise=np.nanmedian(v['ket'][0]),
                 pre_decay=np.nanmedian(v['pre'][1]), ket_decay=np.nanmedian(v['ket'][1])) for r, v in RES4H.items()]
REG4H = pd.DataFrame(reg_rows).sort_values('pre_decay')
print(REG4H.to_string())
print('regions with shorter decay on ketamine:', int((REG4H.ket_decay < REG4H.pre_decay).sum()), 'of', len(REG4H),
      '; with earlier rise:', int((REG4H.ket_rise < REG4H.pre_rise).sum()), ', later rise:', int((REG4H.ket_rise > REG4H.pre_rise).sum()))

**Result: partially reproduced.** Over 36 regions (peak d' of at least 1.3) decay times shorten on ketamine: median change -0.189 s, interval of the median -0.231 to -0.150 s, 34 of 36 regions shorter; with d' of at least 2 (30 regions) -0.186 s. Rise times change by a median of -0.009 s, a twentieth of the decay change, but the interval of the median (-0.018 to -0.002 s) excludes zero and 29 of 36 regions rise earlier, so the rise condition fails under this reading. Under the other reading of the interval (the code's pooled distribution over regions and bootstrap samples) the rise interval includes zero (-0.083 to 0.068) but so does the decay interval (-0.486 to 0.088). The subcortical-to-frontal ordering of decay times is not part of the criterion and is not judged.

## fig5g-mouse-first-order-model

> A two-phase first-order model in which only the persistence time constant differs between preinfusion and ketamine fits the emotion-like activity closely, and varying only that constant captures the dose dependence of affective eye closure for ketamine and PCP; alternative models varying the input scale or the broadcast constant fit worse.

Neural part: the model of the Methods (code model_system) fitted with curve_fit (parameters in [0, 100], condition multipliers in [0, 1]) to the across-session mean emotion-like signal of fig5e, preinfusion and ketamine concatenated (-0.25 to 23 s and 25 to 70 s after the first puff); variance explained 1 - MSE(model)/MSE(null), the null being the same model with no parameter varying across conditions. Alternatives: the broadcast constant or the input scale varies instead. The ketamine-epoch variance explained is the same ratio on the ketamine half.

In [ ]:
cid = 'fig5g-mouse-first-order-model'
rows = []
for key in (MAIN5E, CODE5E):
    d, sig, tt = TAB5E[key]
    ok = ~np.isnan(sig['pre'][0])
    data = np.hstack([sig['pre'][:, ok].mean(0), sig['ket'][:, ok].mean(0)])
    n = len(data) // 2
    for kind in ('persist', 'broadcast', 'saturation'):
        r = m.compare_model(kind, data)
        ve_ket = 1 - np.mean((data[n:] - r['pred'][n:]) ** 2) / np.mean((data[n:] - r['pred_null'][n:]) ** 2)
        rows.append(dict(dimension=f'{key[0]} s, var {key[1]}', model=kind, varexp=r['varexp'], varexp_ketamine_half=ve_ket,
                         S=r['params'][0], tB=r['params'][1], tP=r['params'][2], c_pre=r['params'][3], c_ket=r['params'][4]))
TAB5G = pd.DataFrame(rows); print(TAB5G.to_string())
v, tol = 0.954, 0.095
print(f"neural fit, main dimension: {TAB5G.varexp.iloc[0]:.3f} (expected {v} ± {tol}; within: {within(TAB5G.varexp.iloc[0], v, tol)})")

Behavioural part: eye closure of the dose experiments in DANDI:001347 (`sub-mouse_ses-<id>_behavior.nwb`). Which sessions received which dose is not in the dataset; the lists are read at run time from the paper's code record (Zenodo 14920570, `mouse_eyepuff/scripts/fig1_dose.py`, `experiments`), and that script's `just_ket` and `just_pcp` orderings (ketamine 6, 13, 25, 35 mg/kg; PCP 1, 5, 10 mg/kg) are taken as the conditions of Fig. 5H and 5I. Variants add saline and the highest dose.

In [ ]:
import io, zipfile, hashlib, ast, urllib.request
url = 'https://zenodo.org/api/records/14920570/files/kauvar-richman-liu-2025_zenodo.zip/content'
raw = urllib.request.urlopen(url, timeout=120).read()
print('md5', hashlib.md5(raw).hexdigest(), 'expected bd1fb567121aaf0291337783df57bcfc')
assert hashlib.md5(raw).hexdigest() == 'bd1fb567121aaf0291337783df57bcfc'
zf = zipfile.ZipFile(io.BytesIO(raw))
src = zf.read([n for n in zf.namelist() if n.endswith('mouse_eyepuff/scripts/fig1_dose.py')][0]).decode()
DOSE = None
for node in ast.walk(ast.parse(src)):
    if isinstance(node, ast.Assign) and getattr(node.targets[0], 'id', None) == 'experiments':
        DOSE = ast.literal_eval(node.value)
print(DOSE)
t0 = time.time()
DTR, DMICE = {}, {}
for k, ids in DOSE.items():
    if k != 'kx':
        DTR[k], DMICE[k] = m.dose_trace(ids)
print(f'{sum(len(v) for k, v in DOSE.items() if k != "kx")} sessions read in {time.time() - t0:.0f} s')
print('late eye closure 0.3-0.8 s after the last puff (mean of mice):',
      {k: round(float(v[100 + 7 * 300 + 30:100 + 7 * 300 + 80].mean()), 3) for k, v in DTR.items()})
rows = []
for name, keys in {'ketamine (code just_ket)': ['ket6', 'ket13', 'ket25', 'ket35'],
                   'PCP (code just_pcp)': ['pcp1', 'pcp5', 'pcp10'],
                   'ketamine + 50': ['ket6', 'ket13', 'ket25', 'ket35', 'ket50'],
                   'PCP + 20': ['pcp1', 'pcp5', 'pcp10', 'pcp20'],
                   'ketamine + saline + 50': ['sal', 'ket6', 'ket13', 'ket25', 'ket35', 'ket50'],
                   'PCP + saline + 20': ['sal', 'pcp1', 'pcp5', 'pcp10', 'pcp20']}.items():
    r = m.fit_behaviour([DTR[k] for k in keys])
    rows.append(dict(fit=name, varexp=r['varexp'], S=r['params'][0], tB=r['params'][1], tP=r['params'][2],
                     multipliers=np.round(r['params'][3:], 3).tolist()))
TAB5H = pd.DataFrame(rows); print(TAB5H.to_string())
vk, vp = TAB5H.varexp.iloc[0], TAB5H.varexp.iloc[1]
for a, (ek, ep) in {'ketamine 0.927, PCP 0.870': (0.927, 0.870), 'ketamine 0.870, PCP 0.927': (0.870, 0.927)}.items():
    print(f'assignment {a}: ketamine {vk:.3f} within {within(vk, ek, 0.1 * ek)}, PCP {vp:.3f} within {within(vp, ep, 0.1 * ep)}')

**Result: reproduced.** The persistence model explains 0.881 of the variance of the across-session emotion-like signal of the main dimension (0.947 with the code's window), within 0.954 ± 0.095; letting the broadcast constant or the input scale vary instead explains less (0.871 and 0.871 overall, 0.896 and 0.897 against 0.918 on the ketamine half; 0.941, 0.941 against 0.947 with the code's window). The dose fits give 0.930 for ketamine (6, 13, 25, 35 mg/kg) and 0.870 for PCP (1, 5, 10 mg/kg), within 10% of 0.927 and 0.870 under the assignment of 0.927 to ketamine (Fig. 5H) and 0.870 to PCP (Fig. 5I); the 10% tolerance also admits the other assignment (0.930 against 0.870, 0.870 against 0.927), so the panels cannot be told apart this way. Adding saline and the highest dose lowers both (0.885 and 0.884).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

| claim | outcome | main numbers |
|---|---|---|
| fig6d-mouse-persistent-activity-reduced | reproduced (assumed epochs) | 0.867, -0.002, 0.515; corrected p 0.0005 and 0.029 |
| fig6f-mouse-fast-activity-unchanged | not reproduced (absence claim; the ketamine part holds) | pre vs ketamine corrected p 0.67; ketamine vs post 0.0045 |
| fig5e-mouse-emotion-dimension-late-reduced | not reproduced | late corrected p 0.086 (5-6 s window); 0.039 with the code's 3-4 s window |
| fig6h-mouse-timescale-shorter | reproduced | 0.437 to 0.162 s, p 0.0057 |
| fig6j-mouse-coupling-reduced | not reproduced | ratio 0.852, p 0.30 (Methods); 0.559, p 8.7e-05 (code) |
| fig5d-mouse-emotion-dimension-accumulates | reproduced | accumulation 0.846 pre, -0.048 ketamine; 10-20 s 0.838 |
| figs34-mouse-tone-persistence-reduced | partially reproduced | fast n.s., persistent and timescale reduced, coupling not reduced (Methods) |
| fig4hi-mouse-rise-unchanged-decay-shortened | partially reproduced | decay -0.189 s (CI below 0); rise -0.009 s (CI excludes 0) |
| fig4g-mouse-slow-clusters-altered | reproduced | Spearman rho 0.47; slowest 3 0.507 vs fastest 3 0.368 |
| fig5g-mouse-first-order-model | reproduced | neural 0.881; ketamine 0.930, PCP 0.870 |

Differences from the paper's method:

- Session selection: mouse 5 is taken as the animal the Methods exclude, the only choice that gives both 11 sessions from 9 mice and 7 tone sessions from 7 mice.
- Session z-scoring of the rates is assumed (the code's driver that makes `all_zfr` is not in the record). It does not affect the z-scored projections of Figs. 6D, 6F and S34.
- Coding-dimension variance: the Methods' per-neuron variance is the main reading, the code's pooled scalar a variant. It decides fig6j and the coupling part of figs34, and moves the means of fig6f and fig5e.
- Epochs: the Methods' trials 22 to 32 and 34 to 40 disagree with Table S3's counts (10 and 6) and the code's slices (23 to 32, 35 to 40); the code's slices are adopted for fig6d, fig6f, fig5e, fig4g and fig4hi.
- Emotion-like dimension: the Methods put the target 5 s after the last puff, the code (as its plotting offsets place it) 3 to 4 s after; the criterion's 5 to 6 s is the main reading. Only the code's window reproduces the paper's Fig. 5E values.
- Timescale and coupling: the code's series (11 to 20 and 23 to 32 for the timescale, 1 to 10 and 23 to 32 for coupling) are used, as Table S3's trial counts agree with them; fits from lag 0 and per-series averaging follow the Methods, the code's lag one and concatenation are variants.
- Tone epochs: split at the injection time (10 preinfusion tone series, as Table S3) as the main reading; the code's split gives 9.
- Clustering: Scanpy 1.12 with leidenalg gives 11 clusters, the paper 12; no cluster is dropped (the code drops its cluster 11 in Fig. 4H, I).
- fig5g dose fits: the conditions are the code's `just_ket` and `just_pcp` dose lists; the dose session lists come from the code record.

What was found about the dataset:

- DANDI:001326 holds 13 sessions from 10 mice, as the paper states; units are pre-curated (all 'good', SNR above 2, more than 500 spikes); no ISI metric is stored.
- Spike times are stored unsorted within every unit.
- The subjects are numbered 0 to 9 with no link to the paper's mouse or experiment identifiers; the link to DANDI:001347's Neuropixels-rig behaviour files was made here from the inter-series intervals (10 of 13 sessions; sub-3 ses-20230225 and both sub-5 sessions have no behaviour file).
- The excluded animal is not named in the paper, the code or the dataset.
- sub-5 ses-20230706 has 2 probes, where the Methods say 3 to 4.